In [49]:
import unittest
import pandas as pd
import pickle
import sys
import numpy as np
import import_ipynb
import intrusiondetection
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score
from keras import callbacks

def zscore_normalization(df, name, mean=None, sd=None):
    if mean is None:
        mean = df[name].mean()

    if sd is None:
        sd = df[name].std()

    df[name] = (df[name] - mean) / sd


# Import functions from the notebook
# from your_notebook import preprocess, traintest_split, training_basic_classifier, train, predict

class TestIntrusionDetection(unittest.TestCase):
    def test_df_shape(self):
        df = pd.read_csv('kddcup.data_10_percent_corrected.csv')
        self.assertEqual(df.shape, (494020, 42))    
    
    def test_value_counts(self):
        df = pd.read_csv('kddcup.data_10_percent_corrected.csv')
        self.assertEqual(df['outcome'].value_counts()[0], 396743)
        self.assertEqual(df['outcome'].value_counts()[1], 97277)
    
    def test_normalization(self):
        df = pd.read_csv('kddcup.data_10_percent_corrected.csv')
        df = intrusiondetection.preprocess(df)
        X = df.drop(columns=["outcome"])
        for label in X.columns:
             with self.subTest(column=label):
                mean = X[label].mean()
                std = X[label].std()
    
    def test_components_drop(self):
        df = pd.read_csv('kddcup.data_10_percent_corrected.csv')
        df = intrusiondetection.preprocess(df)
        self.assertEqual(df.shape, (494020, 40))
    
    def test_gbc_training(self):
        df = pd.read_csv('kddcup.data_10_percent_corrected.csv')
        df = intrusiondetection.preprocess(df)
        X_train, X_test, y_train, y_test = intrusiondetection.traintest_split(df)
        model = GradientBoostingClassifier(random_state=42)
        model.fit(X_train, y_train)
        self.assertIsNotNone(model)
        self.assertTrue(hasattr(model, 'feature_importances_'))
    
    def test_gbc_prediction(self):
        df = pd.read_csv('kddcup.data_10_percent_corrected.csv')
        df = intrusiondetection.preprocess(df)
        X_train, X_test, y_train, y_test = intrusiondetection.traintest_split(df)
        model = GradientBoostingClassifier(random_state=42)
        model.fit(X_train, y_train)
        predictions = model.predict(X_test)
        self.assertEqual(len(predictions), len(X_test))
    
    def test_gbc_accuracy(self):
        df = pd.read_csv('kddcup.data_10_percent_corrected.csv')
        df = intrusiondetection.preprocess(df)
        X_train, X_test, y_train, y_test = intrusiondetection.traintest_split(df)
        model = GradientBoostingClassifier(random_state=42)
        model.fit(X_train, y_train)
        predictions = model.predict(X_test)
        accuracy = accuracy_score(y_test, predictions)
        self.assertGreater(accuracy, 0.9)
    
    def test_nb_training(self):
        df = pd.read_csv('kddcup.data_10_percent_corrected.csv')
        df = intrusiondetection.preprocess(df)
        X_train, X_test, y_train, y_test = intrusiondetection.traintest_split(df)
        model = GaussianNB()
        model.fit(X_train, y_train)
        self.assertIsNotNone(model)
        self.assertTrue(hasattr(model, 'class_prior_'))
    
    def test_nb_prediction(self):
        df = pd.read_csv('kddcup.data_10_percent_corrected.csv')
        df = intrusiondetection.preprocess(df)
        X_train, X_test, y_train, y_test = intrusiondetection.traintest_split(df)
        model = GaussianNB()
        model.fit(X_train, y_train)
        predictions = model.predict(X_test)
        self.assertEqual(len(predictions), len(X_test))
    
    def test_nb_accuracy(self):
        df = pd.read_csv('kddcup.data_10_percent_corrected.csv')
        df = intrusiondetection.preprocess(df)
        X_train, X_test, y_train, y_test = intrusiondetection.traintest_split(df)
        model = GaussianNB()
        model.fit(X_train, y_train)
        predictions = model.predict(X_test)
        accuracy = accuracy_score(y_test, predictions)
        self.assertGreater(accuracy, 0.9)
    
    def test_ann_training(self):
        df = pd.read_csv('kddcup.data_10_percent_corrected.csv')
        df = intrusiondetection.preprocess(df)
        X_train, X_test, y_train, y_test = intrusiondetection.traintest_split(df)
        input_shape = [X_train.shape[1]]
        model = tf.keras.Sequential([
            tf.keras.layers.Dense(units=64, activation='relu', input_shape=input_shape),
            tf.keras.layers.Dense(units=64, activation='relu'),
            tf.keras.layers.Dense(units=1)
        ])
        model.build()
        model.compile(optimizer='adam', loss='mae', metrics=['accuracy'])  
        earlystopping = callbacks.EarlyStopping(monitor="val_loss",
                                                    mode="min",
                                                    patience=5,
                                                    restore_best_weights=True)
        history = model.fit(X_train, y_train, validation_data=(X_test,y_test), batch_size=256, epochs=60,callbacks=[earlystopping], verbose=0)
        self.assertIsNotNone(model)
    
    def test_ann_prediction(self):
        df = pd.read_csv('kddcup.data_10_percent_corrected.csv')
        df = intrusiondetection.preprocess(df)
        X_train, X_test, y_train, y_test = intrusiondetection.traintest_split(df)
        input_shape = [X_train.shape[1]]
        model = tf.keras.Sequential([
            tf.keras.layers.Dense(units=64, activation='relu', input_shape=input_shape),
            tf.keras.layers.Dense(units=64, activation='relu'),
            tf.keras.layers.Dense(units=1)
        ])
        model.build()
        model.compile(optimizer='adam', loss='mae', metrics=['accuracy'])  
        earlystopping = callbacks.EarlyStopping(monitor="val_loss",
                                                    mode="min",
                                                    patience=5,
                                                    restore_best_weights=True)
        history = model.fit(X_train, y_train, validation_data=(X_test,y_test), batch_size=256, epochs=60,callbacks=[earlystopping], verbose=0)
        predictions = (model.predict(X_test) > 0.5).astype("int32")
        self.assertEqual(len(predictions), len(X_test))
    
    def test_ann_accuracy(self):
        df = pd.read_csv('kddcup.data_10_percent_corrected.csv')
        df = intrusiondetection.preprocess(df)
        X_train, X_test, y_train, y_test = intrusiondetection.traintest_split(df)
        input_shape = [X_train.shape[1]]
        model = tf.keras.Sequential([
            tf.keras.layers.Dense(units=64, activation='relu', input_shape=input_shape),
            tf.keras.layers.Dense(units=64, activation='relu'),
            tf.keras.layers.Dense(units=1)
        ])
        model.build()
        model.compile(optimizer='adam', loss='mae', metrics=['accuracy'])  
        earlystopping = callbacks.EarlyStopping(monitor="val_loss",
                                                    mode="min",
                                                    patience=5,
                                                    restore_best_weights=True)
        history = model.fit(X_train, y_train, validation_data=(X_test,y_test), batch_size=256, epochs=60,callbacks=[earlystopping], verbose=0)
        self.assertGreater(history.history['accuracy'][-1], 0.9)
    
    def test_svm_training(self):
        df = pd.read_csv('kddcup.data_10_percent_corrected.csv')
        df = intrusiondetection.preprocess(df)
        X_train, X_test, y_train, y_test = intrusiondetection.traintest_split(df)
        model = SVC(random_state=42)
        model.fit(X_train, y_train)
        self.assertIsNotNone(model)
        self.assertTrue(hasattr(model, 'class_weight_'))
    
    def test_svm_prediction(self):
        df = pd.read_csv('kddcup.data_10_percent_corrected.csv')
        df = intrusiondetection.preprocess(df)
        X_train, X_test, y_train, y_test = intrusiondetection.traintest_split(df)
        model = SVC(random_state=42)
        model.fit(X_train, y_train)
        predictions = model.predict(X_test)
        self.assertEqual(len(predictions), len(X_test))
   
    def test_svm_accuracy(self):
        df = pd.read_csv('kddcup.data_10_percent_corrected.csv')
        df = intrusiondetection.preprocess(df)
        X_train, X_test, y_train, y_test = intrusiondetection.traintest_split(df)
        model = SVC(random_state=42)
        model.fit(X_train, y_train)
        predictions = model.predict(X_test)
        accuracy = accuracy_score(y_test, predictions)
        self.assertGreater(accuracy, 0.9)
    
    def test_logreg_training(self):
        df = pd.read_csv('kddcup.data_10_percent_corrected.csv')
        df = intrusiondetection.preprocess(df)
        X_train, X_test, y_train, y_test = intrusiondetection.traintest_split(df)
        model = LogisticRegression(random_state=42, solver='sag')
        model.fit(X_train, y_train)
        self.assertIsNotNone(model)
        self.assertTrue(hasattr(model, 'intercept_'))
    
    def test_logreg_prediction(self):
        df = pd.read_csv('kddcup.data_10_percent_corrected.csv')
        df = intrusiondetection.preprocess(df)
        X_train, X_test, y_train, y_test = intrusiondetection.traintest_split(df)
        model = LogisticRegression(random_state=42, solver='sag')
        model.fit(X_train, y_train)
        predictions = model.predict(X_test)
        self.assertEqual(len(predictions), len(X_test))
    
    def test_logreg_accuracy(self):
        df = pd.read_csv('kddcup.data_10_percent_corrected.csv')
        df = intrusiondetection.preprocess(df)
        X_train, X_test, y_train, y_test = intrusiondetection.traintest_split(df)
        model = LogisticRegression(random_state=42, solver='sag')
        model.fit(X_train, y_train)
        predictions = model.predict(X_test)
        accuracy = accuracy_score(y_test, predictions)
        self.assertGreater(accuracy, 0.9)
    
    def test_rfc_training(self):
        df = pd.read_csv('kddcup.data_10_percent_corrected.csv')
        df = intrusiondetection.preprocess(df)
        X_train, X_test, y_train, y_test = intrusiondetection.traintest_split(df)
        model = RandomForestClassifier(random_state=42)
        model.fit(X_train, y_train)
        self.assertIsNotNone(model)
        self.assertTrue(hasattr(model, 'max_depth'))
    
    def test_rfc_prediction(self):
        df = pd.read_csv('kddcup.data_10_percent_corrected.csv')
        df = intrusiondetection.preprocess(df)
        X_train, X_test, y_train, y_test = intrusiondetection.traintest_split(df)
        model = RandomForestClassifier(random_state=42)
        model.fit(X_train, y_train)
        predictions = model.predict(X_test)
        self.assertEqual(len(predictions), len(X_test))
    
    def test_rfc_accuracy(self):
        df = pd.read_csv('kddcup.data_10_percent_corrected.csv')
        df = intrusiondetection.preprocess(df)
        X_train, X_test, y_train, y_test = intrusiondetection.traintest_split(df)
        model = RandomForestClassifier(random_state=42)
        model.fit(X_train, y_train)
        predictions = model.predict(X_test)
        accuracy = accuracy_score(y_test, predictions)
        self.assertGreater(accuracy, 0.9)
    
    def test_dtc_training(self):
        df = pd.read_csv('kddcup.data_10_percent_corrected.csv')
        df = intrusiondetection.preprocess(df)
        X_train, X_test, y_train, y_test = intrusiondetection.traintest_split(df)
        model = DecisionTreeClassifier(random_state=42)
        model.fit(X_train, y_train)
        self.assertIsNotNone(model)
        self.assertTrue(hasattr(model, 'max_depth'))
    
    def test_dtc_prediction(self):
        df = pd.read_csv('kddcup.data_10_percent_corrected.csv')
        df = intrusiondetection.preprocess(df)
        X_train, X_test, y_train, y_test = intrusiondetection.traintest_split(df)
        model = DecisionTreeClassifier(random_state=42)
        model.fit(X_train, y_train)
        predictions = model.predict(X_test)
        self.assertEqual(len(predictions), len(X_test))
    
    def test_dtc_accuracy(self):
        df = pd.read_csv('kddcup.data_10_percent_corrected.csv')
        df = intrusiondetection.preprocess(df)
        X_train, X_test, y_train, y_test = intrusiondetection.traintest_split(df)
        model = DecisionTreeClassifier(random_state=42)
        model.fit(X_train, y_train)
        predictions = model.predict(X_test)
        accuracy = accuracy_score(y_test, predictions)
        self.assertGreater(accuracy, 0.9)

if __name__ == '__main__':
    main = TestIntrusionDetection()
    suite = unittest.TestLoader().loadTestsFromTestCase(TestIntrusionDetection)
    unittest.TextTestRunner(verbosity=4,stream=sys.stderr).run(suite)

test_ann_accuracy (__main__.TestIntrusionDetection) ... 

ok
test_ann_prediction (__main__.TestIntrusionDetection) ... 

3088/3088 [==============================] - 3s 1ms/step


ok
test_ann_training (__main__.TestIntrusionDetection) ... ok
test_components_drop (__main__.TestIntrusionDetection) ... ok
test_df_shape (__main__.TestIntrusionDetection) ... ok
test_dtc_accuracy (__main__.TestIntrusionDetection) ... ok
test_dtc_prediction (__main__.TestIntrusionDetection) ... ok
test_dtc_training (__main__.TestIntrusionDetection) ... ok
test_gbc_accuracy (__main__.TestIntrusionDetection) ... ERROR
test_gbc_prediction (__main__.TestIntrusionDetection) ... ERROR
test_gbc_training (__main__.TestIntrusionDetection) ... ERROR
test_logreg_accuracy (__main__.TestIntrusionDetection) ... ERROR
test_logreg_prediction (__main__.TestIntrusionDetection) ... ERROR
test_logreg_training (__main__.TestIntrusionDetection) ... ERROR
test_nb_accuracy (__main__.TestIntrusionDetection) ... ok
test_nb_prediction (__main__.TestIntrusionDetection) ... ok
test_nb_training (__main__.TestIntrusionDetection) ... ok
test_normalization (__main__.TestIntrusionDetection) ... ok
test_rfc_accuracy (__

In [50]:
df = pd.read_csv('kddcup.data_10_percent_corrected.csv')
df = intrusiondetection.preprocess(df)
X_train, X_test, y_train, y_test = intrusiondetection.traintest_split(df)
model = GradientBoostingClassifier(random_state=42)
# model.fit(X_train, y_train)
y_test.value_counts()

ValueError: y contains 1 class after sample_weight trimmed classes with zero weights, while a minimum of 2 classes are required.